# 02. PDF 로드 (data/01_raw → data/02_loaded)

임대차 사례집·가이드·표준계약서 PDF를 마크다운(.md)으로 추출한다. 코퍼스가 세 종류로 갈려 단일 방법으로는 안 된다.

| 페이지/문서 상태 | 처리 | 근거 |
| --- | --- | --- |
| 본문 텍스트 정상 | **PyMuPDF4LLM** 마크다운 | 텍스트 레이어로 충분 |
| 텍스트 없음(스캔) / 표가 큰 이미지 | 렌더 → **OpenAI Vision** 전사 | 표 구조를 마크다운 표로 보존 |
| **텍스트 레이어 깨짐**(ToUnicode 손상 → 숫자 누락·음절 중복) | **문서 전체 Vision** | 추출 텍스트가 신뢰 불가 |

세 번째가 핵심 함정이다. 예: `2021 한국부동산원 사례집`은 글자는 추출되지만 숫자(기간·금액·조항)가 통째로 누락된다. 그래서 **문서 단위로 깨짐을 감지해 전 페이지를 Vision으로 강제**한다.

흐름: **0 설정 → 1 한 개 검증 → 2 중복제거+dry-run(비용·강제Vision 미리보기) → 3 전체 처리 → 4 검증**

> ⚠️ Vision은 `.env`의 **실제 `OPENAI_API_KEY`** 가 있어야 동작한다(placeholder면 401).  
> ⚠️ 페이지당 소액 과금 → **2번 dry-run으로 호출 수를 먼저 확인**하고 3번을 실행한다.

## 0. 설정

In [ ]:
import base64
import hashlib
import json
import os
import re
import time
from pathlib import Path

import fitz  # PyMuPDF
import pymupdf4llm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()  # OPENAI_API_KEY 사용

RAW_DIR = Path("../data/01_raw")
PDF_DIR = RAW_DIR / "pdf"          # PDF 원본 (확장자별 폴더)
LOADED_DIR = Path("../data/02_loaded")
LOADED_DIR.mkdir(parents=True, exist_ok=True)

# 라우팅·Vision 파라미터 (필요시 조정)
MIN_CHARS = 50        # 페이지 텍스트가 이보다 짧으면 스캔으로 보고 Vision
IMG_AREA_RATIO = 0.4  # 페이지의 40% 이상 덮는 이미지가 있으면 표·도식으로 보고 Vision
DPI = 200             # Vision용 페이지 렌더 해상도
VISION_MODEL = "gpt-4o-mini"  # 표 품질이 더 필요하면 gpt-4o
VISION_THROTTLE = 1.0  # Vision 호출 간 최소 간격(초) — 분당 토큰한도(TPM) 회피
VISION_MAX_RETRY = 6   # 429/일시 오류 시 지수 백오프 재시도 횟수

# 깨진 텍스트 레이어 강제 Vision 판정용 (본문 페이지 숫자 밀도)
CORRUPT_DIGIT_RATIO = 0.012  # 본문 숫자 비율이 이보다 낮으면 ToUnicode 손상(숫자 누락)으로 봄
CORRUPT_SKIP_PAGES = 3       # 앞쪽 표지·목차는 숫자 밀도가 달라 표본에서 제외
FORCE_VISION_DOCS: set[str] = set()  # 파일명을 넣으면 무조건 전체 Vision
SKIP_CORRUPT_CHECK = False           # True면 자동 깨짐 감지 끔

print("OPENAI_API_KEY 설정됨:", bool(os.environ.get("OPENAI_API_KEY")))
print("입력:", PDF_DIR.resolve())
print("출력:", LOADED_DIR.resolve())

### 헬퍼 함수

- `file_md5` : 중복 PDF 판별
- `is_corrupted_textlayer` : 본문 표본의 **숫자 밀도**로 ToUnicode 손상(숫자 누락) 감지 → **문서 단위 강제 Vision**
- `page_plan` : 정상 문서의 페이지별 라우팅 판정 (이미지·표 있는 페이지만 Vision)
- `plan_pages` : 문서의 페이지별 method 목록 (API 호출 없음 → dry-run 공용)
- `render_png` / `vision_transcribe` : 페이지 렌더 + Vision 전사
- `load_pdf` : 한 PDF 로드 → 각 페이지에 `text`(로더) + `vision`(표 페이지만) 동시 보관
- `save_all` : **3종 산출물** 저장
  - `{이름}.text.md` — 전 페이지 로더 텍스트 (LLM 없음)
  - `{이름}.tables.json` — 이미지·표 페이지 LLM 전사 리스트 `[{page, markdown}]`
  - `{이름}.final.md` — 텍스트 페이지=로더, 표 페이지=LLM 으로 병합한 최종본

In [ ]:
VISION_PROMPT = (
    "다음 이미지는 한국 임대차 관련 공문서 PDF의 한 페이지다. "
    "페이지 내용을 한국어 마크다운으로 그대로 전사하라.\n"
    "규칙:\n"
    "- 표는 반드시 마크다운 표(| 헤더 | ... |)로 구조를 유지한다.\n"
    "- 숫자(기간·금액·조항번호·날짜)를 빠짐없이 정확히 옮긴다.\n"
    "- 머리글·바닥글·페이지번호·로고 등 장식 요소는 제외한다.\n"
    "- 표지·간지처럼 내용이 거의 없으면 빈 문자열만 출력한다(설명·사과 문구 금지).\n"
    "- 원문에 없는 내용을 추가하거나 요약하지 마라.\n"
    "- 전체를 코드블록(```)으로 감싸지 말고, 설명 없이 전사 결과만 출력한다."
)

# 모델이 빈/표지 페이지에서 내뱉는 거부·빈내용 문구 → 빈 페이지로 처리
_REFUSAL_RE = re.compile(
    r"(죄송하지만|전사할 수 없|처리할 수 없|빈 문자열|빈 페이지|내용이 없|"
    r"cannot (assist|process|transcribe)|no (content|text))"
)


def _safe_filename(name: str) -> str:
    name = re.sub(r'[\\/:*?"<>|]', "", name)
    return re.sub(r"\s+", "_", name.strip())


def _clean_md(md: str) -> str:
    """모델이 전체를 ```...``` 코드펜스로 감싼 경우 벗겨낸다."""
    md = md.strip()
    if md.startswith("```"):
        md = re.sub(r"^```[a-zA-Z]*\n?", "", md)
        md = re.sub(r"\n?```$", "", md.rstrip())
    return md.strip()


def file_md5(path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def is_corrupted_textlayer(doc, skip_pages: int = CORRUPT_SKIP_PAGES,
                           sample_pages: int = 8) -> bool:
    """텍스트 레이어 손상(ToUnicode 매핑 깨짐) 여부를 본문 표본으로 판정.

    이 코퍼스의 깨짐 신호는 '음절 중복'이 아니라 **숫자 누락**이다.
    (예: '위원장 N명 이상 M명 이하' → 숫자가 통째로 사라짐)
    표지·목차(앞 skip_pages장)는 숫자 밀도가 달라 표본에서 제외하고,
    본문에서 한글은 충분한데 숫자 비율이 비정상적으로 낮으면 깨짐으로 본다.
    """
    samples = []
    for idx, page in enumerate(doc):
        if idx < skip_pages:
            continue
        t = page.get_text("text").strip()
        if len(t) >= 100:
            samples.append(t)
        if len(samples) >= sample_pages:
            break
    if not samples:
        return False  # 본문 텍스트가 거의 없음 → 페이지 라우팅(스캔)이 알아서 처리
    joined = "".join(samples)
    hangul = len(re.findall(r"[가-힣]", joined))
    digits = len(re.findall(r"\d", joined))
    return hangul >= 500 and digits / len(joined) < CORRUPT_DIGIT_RATIO


def doc_force_vision(doc, source_name: str) -> bool:
    if source_name in FORCE_VISION_DOCS:
        return True
    if SKIP_CORRUPT_CHECK:
        return False
    return is_corrupted_textlayer(doc)


def page_plan(page, page_area: float):
    """정상 문서의 페이지 라우팅. (needs_vision, text, 최대이미지면적비) 반환."""
    text = page.get_text("text").strip()
    biggest = 0.0
    for im in page.get_image_info():
        x0, y0, x1, y1 = im["bbox"]
        biggest = max(biggest, abs((x1 - x0) * (y1 - y0)) / page_area)
    needs = len(text) < MIN_CHARS or biggest > IMG_AREA_RATIO
    return needs, text, biggest


def plan_pages(doc, force: bool) -> list[str]:
    """페이지별 method('text'|'vision') 목록. Vision 호출 없음 → dry-run 공용.

    - force=True(깨진 사례집): 이미지 유무와 무관하게 전 페이지 vision
    - force=False(일반 문서): 텍스트가 짧거나 큰 이미지가 덮은 페이지만 vision
    """
    if force:
        return ["vision"] * len(doc)
    parea = doc[0].rect.width * doc[0].rect.height
    return ["vision" if page_plan(p, parea)[0] else "text" for p in doc]


def render_png(page, dpi: int = DPI) -> bytes:
    return page.get_pixmap(dpi=dpi).tobytes("png")


def vision_transcribe(png: bytes) -> str:
    """페이지 이미지를 마크다운으로 전사. 429 등은 지수 백오프로 재시도하고,
    빈/표지 페이지의 거부 문구는 빈 안내로 치환하며, 감싼 코드펜스는 제거한다."""
    b64 = base64.b64encode(png).decode()
    for attempt in range(VISION_MAX_RETRY):
        try:
            resp = client.chat.completions.create(
                model=VISION_MODEL,
                temperature=0,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": VISION_PROMPT},
                        {"type": "image_url",
                         "image_url": {"url": f"data:image/png;base64,{b64}",
                                       "detail": "high"}},
                    ],
                }],
            )
            md = _clean_md(resp.choices[0].message.content or "")
            if not md or _REFUSAL_RE.search(md):
                return "<!-- (빈/표지 페이지) -->"
            return md
        except Exception as e:  # noqa: BLE001
            # 429(rate limit)·일시 오류는 백오프 후 재시도, 마지막엔 주석으로 기록
            if attempt == VISION_MAX_RETRY - 1:
                return f"<!-- vision 실패: {e} -->"
            time.sleep(2 ** attempt)  # 1, 2, 4, 8, 16초
    return "<!-- vision 실패: 재시도 초과 -->"


def load_pdf(path, use_vision: bool = True, verbose: bool = True,
             page_range: tuple | None = None):
    """한 PDF를 페이지 단위로 로드. (meta, pages) 반환.

    각 page 레코드는 text와 vision을 **둘 다** 들고 있어 3종 산출물을 만든다:
      {page, method, text, vision}
        - text  : PyMuPDF4LLM 로더 텍스트 (전 페이지 항상 채움)
        - vision: 이미지·표 페이지(또는 깨진 문서)만 LLM 전사, 그 외 None

    page_range: (start, end) 1-based inclusive, e.g. (1, 11). None이면 전 페이지.
    """
    path = Path(path)
    doc = fitz.open(str(path))
    force = doc_force_vision(doc, path.name)
    methods = plan_pages(doc, force)
    # 로더 텍스트는 항상 확보 (깨진 문서도 ①text.md 산출물엔 담는다)
    md_chunks = pymupdf4llm.to_markdown(
        str(path), page_chunks=True, show_progress=False)

    # 처리할 페이지 인덱스 결정 (0-based)
    if page_range:
        s, e = page_range[0] - 1, page_range[1]
        indices = list(range(s, min(e, len(doc))))
    else:
        indices = list(range(len(doc)))

    pages, n_text, n_vis = [], 0, 0
    for i in indices:
        page = doc[i]
        text = (md_chunks[i]["text"].strip()
                if i < len(md_chunks) else page.get_text("text").strip())
        vision = None
        if methods[i] == "vision":
            if use_vision:
                vision = vision_transcribe(render_png(page))
                time.sleep(VISION_THROTTLE)  # TPM 한도 회피
            else:
                vision = "<!-- (vision 보류: use_vision=False) -->"
            n_vis += 1
        else:
            n_text += 1
        pages.append({"page": i + 1, "method": methods[i],
                      "text": text, "vision": vision})
        if verbose:
            print(f"  p{i + 1}/{len(doc)} [{methods[i]}]   ", end="\r")
    doc.close()
    meta = {
        "source": path.name,
        "md5": file_md5(path),
        "num_pages": len(pages),
        "n_text": n_text,
        "n_vision": n_vis,
        "forced_vision": force,
        "page_range": page_range,
    }
    if verbose:
        tag = "  [문서 전체 강제 Vision]" if force else ""
        print(f"  완료: {len(pages)}p (text {n_text} / vision {n_vis}){tag}        ")
    return meta, pages


def _header(meta, kind: str) -> list[str]:
    range_tag = (f" · pages {meta['page_range'][0]}-{meta['page_range'][1]}"
                 if meta.get("page_range") else "")
    return [
        f"# {meta['source']}  ({kind})\n",
        f"> md5: {meta['md5']}  ",
        f"> pages: {meta['num_pages']} (text {meta['n_text']} / vision {meta['n_vision']})"
        f"{' · 강제Vision' if meta.get('forced_vision') else ''}{range_tag}\n",
    ]


def save_text_md(meta, pages, out_dir: Path = LOADED_DIR) -> Path:
    """① 텍스트만 — 전 페이지 로더 텍스트 (LLM 없음)."""
    slug = _safe_filename(os.path.splitext(meta["source"])[0])
    out = _header(meta, "text")
    for p in pages:
        out.append(f"\n<!-- page {p['page']} -->\n")
        out.append(p["text"])
    path = out_dir / f"{slug}.text.md"
    path.write_text("\n".join(out), encoding="utf-8")
    return path


def save_tables_json(meta, pages, out_dir: Path = LOADED_DIR) -> Path:
    """② 표 데이터 — 이미지·표 페이지의 LLM 전사만 리스트로 (page = id)."""
    slug = _safe_filename(os.path.splitext(meta["source"])[0])
    records = [{"page": p["page"], "markdown": p["vision"]}
               for p in pages if p["method"] == "vision" and p["vision"]]
    payload = {
        "source": meta["source"],
        "md5": meta["md5"],
        "forced_vision": meta.get("forced_vision", False),
        "tables": records,
    }
    path = out_dir / f"{slug}.tables.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2),
                    encoding="utf-8")
    return path


def save_final_md(meta, pages, out_dir: Path = LOADED_DIR) -> Path:
    """③ 최종 — 텍스트 페이지는 로더 텍스트, 표 페이지는 LLM 결과로 병합."""
    slug = _safe_filename(os.path.splitext(meta["source"])[0])
    out = _header(meta, "final")
    for p in pages:
        body = p["vision"] if (p["method"] == "vision" and p["vision"]) else p["text"]
        out.append(f"\n<!-- page {p['page']} ({p['method']}) -->\n")
        out.append(body)
    path = out_dir / f"{slug}.final.md"
    path.write_text("\n".join(out), encoding="utf-8")
    return path


def save_all(meta, pages, out_dir: Path = LOADED_DIR) -> dict:
    """3종 산출물 저장: text.md / tables.json / final.md."""
    return {
        "text": save_text_md(meta, pages, out_dir),
        "tables": save_tables_json(meta, pages, out_dir),
        "final": save_final_md(meta, pages, out_dir),
    }

## 1. PDF 한 개 검증

`TEST_PDF` 한 개로 라우팅→로드→.md 저장이 잘 되는지 확인한다. 다른 파일은 `TEST_PDF`만 바꿔 재실행.

> 비용 없이 구조만 보려면 `load_pdf(TEST_PDF, use_vision=False)` 로 바꿔 실행.

In [ ]:
# 검증할 파일 하나 선택
TEST_PDF = PDF_DIR / "2021_[한국부동산원] 주택·상가건물 임대차분쟁조정사례집.pdf"
TEST_PAGES = (1, 11)  # 테스트 페이지 범위 (1-based inclusive). None이면 전체.

# (1) API 호출 없이 라우팅/강제Vision 여부 미리 확인
_doc = fitz.open(str(TEST_PDF))
_force = doc_force_vision(_doc, TEST_PDF.name)
_methods = plan_pages(_doc, _force)
_doc.close()
_test_idx = (list(range(TEST_PAGES[0] - 1, TEST_PAGES[1])) if TEST_PAGES
             else list(range(len(_methods))))
_v = sum(1 for i in _test_idx if _methods[i] == "vision")
print(f"강제Vision: {_force}  |  테스트범위: {TEST_PAGES}  |  Vision 예정: {_v}회")

# (2) 실제 로드 + 3종 산출물 저장  (구조만 보려면 use_vision=False)
meta, pages = load_pdf(TEST_PDF, use_vision=True, page_range=TEST_PAGES)
outs = save_all(meta, pages)
for kind, p in outs.items():
    print(f"저장 [{kind:6}]: {p.resolve()}")

# (3) 미리보기 — 텍스트 페이지 / 표(vision) 페이지 각 한 장
_t = next((p for p in pages if p["method"] == "text"), None)
_vp = next((p for p in pages if p["method"] == "vision" and p["vision"]), None)
if _t:
    print(f"\n----- p{_t['page']} [text] (앞 600자) -----\n{_t['text'][:600]}")
if _vp:
    print(f"\n----- p{_vp['page']} [vision] (앞 600자) -----\n{_vp['vision'][:600]}")

## 2. 중복 제거 + dry-run (비용 미리보기)

md5 동일 PDF는 하나만 남기고, 파일별 vision 호출 수와 강제Vision 여부를 **실제 호출 없이** 집계한다.

In [ ]:
# md5 중복 제거
seen, unique = {}, []
for f in sorted(PDF_DIR.glob("*.pdf")):
    h = file_md5(f)
    if h in seen:
        print(f"중복 skip: {f.name}\n        == {seen[h].name}")
    else:
        seen[h] = f
        unique.append(f)

print(f"\n고유 PDF {len(unique)}개\n")

# 파일별 라우팅 집계
total_text = total_vis = 0
_methods_map = {}  # 팀 분배 셀에서 참조
print(f"{'파일':43} | {'text':>5} | {'vision':>6} | 강제")
print("-" * 70)
for f in unique:
    doc = fitz.open(str(f))
    force = doc_force_vision(doc, f.name)
    methods = plan_pages(doc, force)
    doc.close()
    t, v = methods.count("text"), methods.count("vision")
    _methods_map[f.name] = {"text": t, "vision": v, "force": force}
    total_text += t
    total_vis += v
    print(f"{f.name[:41]:43} | {t:>5} | {v:>6} | {'■' if force else ''}")
print("-" * 70)
print(f"{'합계':43} | {total_text:>5} | {total_vis:>6} |")
print(f"\n→ Vision 호출 총 {total_vis}회 예정 (모델 {VISION_MODEL})")

# 비용 추정 (gpt-4o-mini: $0.15/1M input, $0.60/1M output)
_input_tok  = total_vis * 1_500
_output_tok = total_vis * 500
est_cost = _input_tok / 1e6 * 0.15 + _output_tok / 1e6 * 0.60
print(f"\n💰 예상 비용 (gpt-4o-mini): ~${est_cost:.3f}  "
      f"(Vision {total_vis}회 × 평균 ~2,000 tokens/page)")

## 3. 전체 처리

고유 PDF를 모두 로드해 `data/02_loaded/`에 파일당 **3종**(`.text.md` / `.tables.json` / `.final.md`)으로 저장한다.
**이미 `.final.md`가 있으면 건너뛴다** — 중단 후 재실행하면 남은 파일만 처리(비용 절약).

In [ ]:
for f in unique:
    slug = _safe_filename(f.stem)
    if (LOADED_DIR / f"{slug}.final.md").exists():
        print(f"skip(이미 있음): {slug}.final.md")
        continue
    print(f"처리 중: {f.name}")
    meta, pages = load_pdf(f)
    save_all(meta, pages)
    print(f"  → {slug}.{{text.md, tables.json, final.md}}  "
          f"(text {meta['n_text']} / vision {meta['n_vision']})\n")

print("완료")

## 4. 검증

파일당 `.final.md`와 `.tables.json`을 확인한다. 추출 불가/깨짐이던 사례집(2021·2022·2023)이 0바이트가 아닌지, 표 페이지 수가 잡혔는지 본다.

In [ ]:
finals = sorted(LOADED_DIR.glob("*.final.md"))
print(f"생성된 final.md {len(finals)}개  (파일당 text.md·tables.json 동반)\n")
print(f"{'파일':52} {'final':>9} | {'표페이지':>6}")
print("-" * 76)
for p in finals:
    kb = p.stat().st_size / 1024
    tbl = p.with_name(p.name[:-len(".final.md")] + ".tables.json")
    n_tbl = (len(json.loads(tbl.read_text(encoding="utf-8")).get("tables", []))
             if tbl.exists() else 0)
    flag = "  ⚠️ 비어있음" if p.stat().st_size < 100 else ""
    print(f"{p.name[:50]:52} {kb:7.1f}KB | {n_tbl:>6}{flag}")